<a href="https://colab.research.google.com/github/Satyadeep-Dey/genai-lab/blob/main/6_Pipelines_%26_low_level_APIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipelines

A Hugging Face pipeline is a high-level API that simplifies the use of pretrained models for various machine learning tasks like text generation, translation, summarization, image classification, and more.

It abstracts away the complexities of loading models, tokenizing inputs, and processing outputs, allowing users to perform tasks with just a few lines of code.

This makes it easy to work with models without needing deep knowledge of their internal workings.

Hugging Face pipelines primarily support transformers, but they also have Diffusers Pipelines for generative models like Stable Diffusion. Diffusers use a similar API but are optimized for image generation and related tasks.

## Note
when you use a Hugging Face pipeline for the first time, it will automatically download the default or specified pretrained model associated with the task from the Hugging Face Hub. Subsequently, it will be cached for faster use in future runs.

Under the hood it does the following :

1. Downloads the tokenizer and model weights from Hugging Face Hub (if not already cached).
2. Loads them into memory.
3. Returns a ready-to-use pipeline object.


In [ ]:
!pip install -q transformers datasets diffusers soundfile accelerate

#Transformers library from Hugging Face provides state-of-the-art pretrained models for NLP and other AI tasks
#Diffusers library from Hugging Face focuses on diffusion models for generative tasks like image and audio synthesis.
#Datasets library from Hugging Face -> easily accessing and sharing datasets for Audio, Computer Vision, and NLP tasks


In [ ]:
# Import necessary libraries

import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline,AutoTokenizer, AutoModelForTokenClassification
from transformers import AutoModelForQuestionAnswering,AutoModelForSeq2SeqLM, AutoConfig
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech,SpeechT5HifiGan
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio
import numpy as np


In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)
# add_to_git_credential=True : tells Hugging Face to also save your token in Git credentials, so you can:
# push models, pull private models, clone private repos, use Git with Hugging Face
# without logging in again.

In [ ]:
# Sentiment Analysis

model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
classifier = pipeline("sentiment-analysis", model=model_name, device="cuda")  # device=0 for GPU (CUDA)


In [ ]:
review = """On the day of arrival (!), our Deluxe Room Booking Referral was suddenly cancelled due to overbooking by the hotel, according to Booking.com. We only found out about it when we were standing in front of the front desk manager who was talking about AC problems. From him we received some lame excuses, lots of hot air and a glass of water, but no alternative room or other support.
In the afternoon in the Hochsaisen in Lisbon without a room, I really don't begrudge anyone.
Not recommended!!!"""
result = classifier(review)
print(result)

In [ ]:
review = """This boutique hotel is very unassuming but has the most stunning oasis surrounding its private pool, right in the centre of Lisbon downtown.
The staff are their next biggest asset, in particular in the restaurant and poolside, where literally nothing is too much trouble and everything is delicious.
We had a room with a huge terrace, complete with reclining deckchairs that still had sun, long after it had left the pool!"""
result = classifier(review)
print(result)


In [ ]:
review = """ We stayed at the hotel for four days in July and have had mixed experiences. We found the outdoor area positive.
The garden was very nice and well maintained and also the pool was clean.
The room was appealing but had small flaws (the windows could not close properly, so it was noisy, facing the garden) and the shower
always provided a small flood as it did not seal properly.
With the high prices per night, we are especially not satisfied with the price-performance ratio. The breakfast offer at the
buffet could have been richer, e.g. in bread selection and fruit.
However , the service was very good and the the staff was very polite and helpful.
"""
result = classifier(review)
print(result)

# This is a mixed review and the model does not perform as well as "gpt-4.1-mini"


In [ ]:
#Text Classification using pipeline . Can also be used for sentiment analysis .

# Load the Text Classification pipeline
#classifier = pipeline("text-classification", model="nlptown/bert-base-multilingual-uncased-sentiment")
classifier = pipeline("text-classification", model="nlptown/bert-base-multilingual-uncased-sentiment", device=0)



In [ ]:
review = """On the day of arrival (!), our Deluxe Room Booking Referral was suddenly cancelled due to overbooking by the hotel, according to Booking.com. We only found out about it when we were standing in front of the front desk manager who was talking about AC problems. From him we received some lame excuses, lots of hot air and a glass of water, but no alternative room or other support.
In the afternoon in the Hochsaisen in Lisbon without a room, I really don't begrudge anyone.
Not recommended!!!"""
# Run classification
result = classifier(review)
print(result)


In [ ]:
# Sample text to classify
text = "Im not sure if I like this AI technology . I don't hate it either"
# Run classification
result = classifier(text)
print(result)


In [ ]:
# Sample text to classify
text = "I absolutely hate this new AI technology! It's evil!!"
# Run classification
result = classifier(text)
# Print result
print(result)


In [ ]:
review = """ We stayed at the hotel for four days in July and have had mixed experiences. We found the outdoor area positive.
The garden was very nice and well maintained and also the pool was clean.
The room was appealing but had small flaws (the windows could not close properly, so it was noisy, facing the garden) and the shower
always provided a small flood as it did not seal properly.
With the high prices per night, we are especially not satisfied with the price-performance ratio. The breakfast offer at the
buffet could have been richer, e.g. in bread selection and fruit.
However , the service was very good and the the staff was very polite and helpful.
"""
result = classifier(review)
# Print result
print(result)

#Expected Output:#
The output will depend on the model used.If using a sentiment analysis model, you might get

`[{'label': '5 stars', 'score': 0.95}]`

This means the model classified the text as positive sentiment (5 stars).

To use GPU (CUDA) for faster processing, modify the pipeline:

`classifier = pipeline("text-classification", model="nlptown/bert-base-multilingual-uncased-sentiment", device=0)`

#Try different models for various classification tasks:#

* "distilbert-base-uncased-finetuned-sst-2-english" (binary sentiment
classification)
* "facebook/bart-large-mnli" (multi-label classification)
* "cardiffnlp/twitter-roberta-base-offensive" (offensive language detection)



In [ ]:
# Named Entity Recognition (NER)
# Model taken from Hugging Face
# https://huggingface.co/dslim/distilbert-NER

tokenizer = AutoTokenizer.from_pretrained("dslim/distilbert-NER")
model = AutoModelForTokenClassification.from_pretrained("dslim/distilbert-NER")
ner_distilbert = pipeline("ner", model=model, tokenizer=tokenizer)


In [ ]:
example = "My name is Wolfgang and I live in Berlin"

# Run the NER model
entities = ner_distilbert(example)
# Print results
for entity in entities:
    print(entity)

# result is quiet good
# [{'entity': 'B-PER', 'score': np.float32(0.99110633), 'index': 4, 'word': 'Wolfgang', 'start': 11, 'end': 19},
#  {'entity': 'B-LOC', 'score': np.float32(0.9967968), 'index': 9, 'word': 'Berlin', 'start': 34, 'end': 40}]


In [ ]:
example = "My name is Sachin and I live in India. I played cricket for Mumbai and India"

# Run the NER model
entities = ner_distilbert(example)
# Print results
for entity in entities:
    print(entity)

# Note : results are not that good for Indian names !!
# [{'entity': 'B-PER', 'score': np.float32(0.9969759), 'index': 4, 'word': 'Sa', 'start': 11, 'end': 13},
#  {'entity': 'B-PER', 'score': np.float32(0.99672407), 'index': 5, 'word': '##chin', 'start': 13, 'end': 17},
#  {'entity': 'B-LOC', 'score': np.float32(0.9986732), 'index': 10, 'word': 'India', 'start': 32, 'end': 37},
#  {'entity': 'B-LOC', 'score': np.float32(0.99781287), 'index': 16, 'word': 'Mumbai', 'start': 60, 'end': 66},
#  {'entity': 'B-LOC', 'score': np.float32(0.9981446), 'index': 18, 'word': 'India', 'start': 71, 'end': 76}]



In [ ]:
# Let's try another model for Named Entity Recognition (NER)
#Model taken from Hugging Face
# Roberta : https://huggingface.co/AventIQ-AI/roberta-named-entity-recognition

# Load the Named Entity Recognition (NER) pipeline
ner_bert_large = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english")
#model="AventIQ-AI/roberta-named-entity-recognition")


In [ ]:
example = "My name is Wolfgang and I live in Berlin"

# Run the NER model
entities = ner_bert_large(example)
# Print results
for entity in entities:
    print(entity)

In [ ]:
example = "My name is Sachin and I live in India. I played cricket for Mumbai and India"

# Run the NER model
entities = ner_bert_large(example)
# Print results
for entity in entities:
    print(entity)

# Note : results are not that good for Indian names !!
# [{'entity': 'B-PER', 'score': np.float32(0.9969759), 'index': 4, 'word': 'Sa', 'start': 11, 'end': 13},
#  {'entity': 'B-PER', 'score': np.float32(0.99672407), 'index': 5, 'word': '##chin', 'start': 13, 'end': 17},
#  {'entity': 'B-LOC', 'score': np.float32(0.9986732), 'index': 10, 'word': 'India', 'start': 32, 'end': 37},
#  {'entity': 'B-LOC', 'score': np.float32(0.99781287), 'index': 16, 'word': 'Mumbai', 'start': 60, 'end': 66},
#  {'entity': 'B-LOC', 'score': np.float32(0.9981446), 'index': 18, 'word': 'India', 'start': 71, 'end': 76}]


In [ ]:
# Question Answering with Context
# Load the Question Answering pipeline
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")


# Define context
context = """
Mahishmathi is a fictitous kingdom from the movie Bahubali . It was ruled by an evil king called Bhallal Dev.
He deposed of his mother whose name was Shivagami Devi . The evil king was defeated by Bahubali whose real name was Shiva"
"""
# Define question
question = "What was the name of the king ?"

# Run the QA model
answer = qa_pipeline(question=question, context=context)

# Print result
print("Answer:", answer["answer"])

# Define question
question = "What was the name of the kingdom ?"

# Run the QA model
answer = qa_pipeline(question=question, context=context)

# Print result
print("Answer:", answer["answer"])

# Define question
question = "Who rescued the kingdom  ?" # a bit if inference !

# Run the QA model
answer = qa_pipeline(question=question, context=context)

# Print result
print("Answer:", answer["answer"])

In [ ]:
# Text Generation using pipeline

# Load the Text Generation pipeline
generator = pipeline("text-generation", model="gpt2")

# Provide a prompt
prompt = "Artificial Intelligence is transforming the world by"

# Generate text
generated_text = generator(prompt, max_length=50, num_return_sequences=1)

# Print result
print("Generated Text:", generated_text[0]["generated_text"])


In [ ]:
# Generate image
image_gen = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16"
).to("cuda")

text = "A colourful bird in the surreal style of Salvador Dali"
image = image_gen(prompt=text).images[0]

image.save("bird.png") # saves the file in the current working directory
# will be deleted once I delete runtime

from google.colab import files
files.download("bird.png") # downloads to local computer

🏎️ **Real-World Production Code NEVER Uses Pipeline**

That’s because it's not stable . A code that works today may not work after a few months due to incompatible changes .

Pipeline hides all this complexity — until it breaks.

🧩**The Core Concept**

pipeline() is a convenience wrapper.

AutoTokenizer + AutoModel + generate() is the real engine.

pipeline = training wheels.

In [ ]:
# Generate audio
# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load processor and model directly (more stable than pipeline)
processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts").to(device)
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to(device)

# we create a random but valid speaker embedding (512-dim)
speaker_embedding = torch.randn(1, 512).to(device)

# Prepare input text
text = "Hi there. How is life treating you .I hope all is well !!"

inputs = processor(text=text, return_tensors="pt").to(device)

# Generate speech
speech = model.generate_speech(
    inputs["input_ids"],
    speaker_embedding,
    vocoder=vocoder
)

# Save audio
sf.write("speech.wav", speech.cpu().numpy(), 16000)
Audio("speech.wav")


In [ ]:
# Text Summarization
# Not using pipelines anymore because it changes and then old code breaks
# Using the API directly

model_name = "facebook/bart-large-cnn"

text = """The Ramayana also known as Valmiki Ramayana, as traditionally attributed to Valmiki, is a smriti text (also described as a Sanskrit epic)
from ancient India, one of the two important epics of Hinduism known as the Itihasas, the other being the Mahabharata.
The epic narrates the life of Rama, the seventh avatar of the Hindu deity Vishnu, who is a prince of Ayodhya in the kingdom of Kosala.
The epic follows his fourteen-year exile to the forest urged by his father King Dasharatha, on the request of Rama's stepmother Kaikeyi;
his travels across the forests in the Indian subcontinent with his wife Sita and brother Lakshmana; the kidnapping of Sita by Ravana, the king of Lanka, that resulted in bloodbath;
and Rama's eventual return to Ayodhya along with Sita to be crowned as a king amidst jubilation and celebration.
"""

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

inputs = tokenizer(text, return_tensors="pt", truncation=True)

summary_ids = model.generate(
    inputs["input_ids"],
    max_new_tokens=60,
    min_new_tokens=20,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary)


In [ ]:
#Unpack this dictionary into keyword arguments.”

book1 = {"title": "Great Expectations", "author": "Charles Dickens","language":"English"}
print(book1)

def describe_book(title, author, language):
    print(f"{title} was written by {author} in {language}.")

describe_book(**book1) #Unpack this dictionary into keyword arguments.”
# this is equivalent to
describe_book(
    title="Great Expectations",
    author="Charles Dickens",
    language="English"
)



In [ ]:
#Translation of text
# Not using pipelines anymore because it changes and then old code breaks
# Using the API directly

model_name = "facebook/nllb-200-distilled-600M"

# Load config first
config = AutoConfig.from_pretrained(model_name)

# Fix the warning
config.tie_word_embeddings = False

# Load tokenizer and model with modified config
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    config=config
).to("cuda")

# Set source language
tokenizer.src_lang = "eng_Latn"

text = "Bangalore is the capital of Karnataka which is in Southern India "

inputs = tokenizer(text, return_tensors="pt").to("cuda")

# Convert target language token properly (NEW way)
target_lang_id = tokenizer.convert_tokens_to_ids("hin_Deva")

translated_tokens = model.generate(
    **inputs, #Unpack this dictionary into keyword arguments.”
    forced_bos_token_id=target_lang_id,
    max_length=128
)

output = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)

print("Hindi:", output[0])
